In [4]:
import pandas as pd

# Load files
gold = pd.read_csv('Gold Price.csv').sort_values('Date')
gold['Date'] = pd.to_datetime(gold['Date'])
if gold['Price'].dtype == 'O': gold['Price'] = gold['Price'].str.replace(',', '').astype(float)

usd = pd.read_csv('USD_INR Historical Data (1).csv').sort_values('Date')
usd['Date'] = pd.to_datetime(usd['Date'], dayfirst=True)
if usd['Price'].dtype == 'O': usd['Price'] = usd['Price'].str.replace(',', '').astype(float)

cpi = pd.read_csv('CPIndex_Jan15-To-Jan25.csv', skiprows=1)
cpi['Date'] = pd.to_datetime(cpi['Year'].astype(str) + '-' + cpi['Month'], format='%Y-%B')

# Merge into the master dataframe 'df'
df = pd.merge(gold[['Date', 'Price']], usd[['Date', 'Price']], on='Date', how='inner', suffixes=('_Gold', '_USD'))
cpi_daily = pd.merge(pd.DataFrame({'Date': pd.date_range(df['Date'].min(), df['Date'].max(), freq='D')}),
                     cpi[['Date', 'Combined Inflation']], on='Date', how='left').ffill()
df = pd.merge(df, cpi_daily, on='Date', how='inner').dropna().sort_values('Date')

print("✅ Step 1 Complete: Master Dataframe 'df' created.")
print(df.tail(3))

✅ Step 1 Complete: Master Dataframe 'df' created.
           Date  Price_Gold  Price_USD  Combined Inflation
2820 2025-09-30      116573     88.841               26.37
2821 2025-10-01      116968     88.683               26.37
2822 2025-10-03      117370     88.747               26.37


In [10]:
import numpy as np
from sklearn.preprocessing import MinMaxScaler

# 1. Calculate Log Returns (The key fix for the downward drift)
df['Gold_Return'] = np.log(df['Price_Gold'] / df['Price_Gold'].shift(1))
df = df.dropna()

# 2. Define Features (Using returns instead of raw price)
# We keep USD and CPI as context
features_to_scale = ['Gold_Return', 'Price_USD', 'Combined Inflation']
scaler = MinMaxScaler(feature_range=(-1, 1)) # Returns can be negative, so -1 to 1 is better
scaled_data = scaler.fit_transform(df[features_to_scale])

# 3. Create Sequences
seq_length = 30
X, y = [], []

for i in range(seq_length, len(scaled_data)):
    X.append(scaled_data[i-seq_length:i])
    y.append(scaled_data[i, 0]) # Target is the Gold Return

X, y = np.array(X), np.array(y)
print(f"✅ Step 2 (Returns-Based) Complete. Shape: {X.shape}")

✅ Step 2 (Returns-Based) Complete. Shape: (2730, 30, 3)


In [11]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
import tensorflow as tf

model_lstm = Sequential([
    LSTM(64, return_sequences=True, input_shape=(X.shape[1], X.shape[2])),
    BatchNormalization(), # Helps stabilize the learning of returns
    Dropout(0.3),

    LSTM(32, return_sequences=False),
    Dropout(0.3),

    Dense(16, activation='relu'),
    Dense(1) # Predicts the Return
])

model_lstm.compile(optimizer='adam', loss=tf.keras.losses.Huber())
print("Training Model on Price Momentum...")
model_lstm.fit(X, y, batch_size=32, epochs=25, verbose=1)
print("✅ Step 3 Complete: Return-based model trained.")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Training Model on Price Momentum...
Epoch 1/25
86/86 ━━━━━━━━━━━━━━━━━━━━ 8s 32ms/step - loss: 0.0394
Epoch 2/25
86/86 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - loss: 0.0162
Epoch 3/25
86/86 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - loss: 0.0146
Epoch 4/25
86/86 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - loss: 0.0140
Epoch 5/25
86/86 ━━━━━━━━━━━━━━━━━━━━ 4s 41ms/step - loss: 0.0131
Epoch 6/25
86/86 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - loss: 0.0121
Epoch 7/25
86/86 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - loss: 0.0117
Epoch 8/25
86/86 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - loss: 0.0115
Epoch 9/25
86/86 ━━━━━━━━━━━━━━━━━━━━ 3s 38ms/step - loss: 0.0111
Epoch 10/25
86/86 ━━━━━━━━━━━━━━━━━━━━ 3s 34ms/step - loss: 0.0114
Epoch 11/25
86/86 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - loss: 0.0113
Epoch 12/25
86/86 ━━━━━━━━━━━━━━━━━━━━ 3s 30ms/step - loss: 0.0116
Epoch 13/25
86/86 ━━━━━━━━━━━━━━━━━━━━ 3s 29ms/step - loss: 0.0118
Epoch 14/25
86/86 ━━━━━━━━━━━━━━━━━━━━ 4s 43ms/step - loss: 0.0110
Epoch 15/25
86/86 ━━━━━━━━━━━━━━━━━

In [17]:
#@title 📅 Enter Future Date (2026/2027)
target_date_input = "2026-02-01" #@param {type:"string"}
target_date = pd.to_datetime(target_date_input)

def get_momentum_prediction_fixed(target_date):
    # Use the absolute latest price and date from your actual dataframe
    last_real_price = df['Price_Gold'].iloc[-1]
    last_date = df['Date'].max()
    days_to_go = (target_date - last_date).days

    if days_to_go <= 0:
        return f"Date already passed. Price on {last_date.date()} was {last_real_price:.2f} INR"

    # Ensure window is correctly shaped [1, 30, 3]
    current_window = scaled_data[-seq_length:].reshape(1, seq_length, 3)
    predicted_price = last_real_price


    for _ in range(days_to_go):
        # 1. Predict Return
        pred_scaled = model_lstm.predict(current_window, verbose=0)

        # 2. Extract scalar value using .item() to stop DeprecationWarnings
        pred_return_val = pred_scaled.item()

        # 3. Inverse scale to get the actual Log Return
        # We need a dummy row of 3 features for the scaler
        dummy = np.zeros((1, 3))
        dummy[0, 0] = pred_return_val
        actual_return = scaler.inverse_transform(dummy)[0, 0]

        # 4. Update the Price
        predicted_price = predicted_price * np.exp(actual_return)

        # 5. Slide Window
        # Keep USD and CPI constant from the last known state
        last_usd = current_window[0, -1, 1]
        last_cpi = current_window[0, -1, 2]

        next_step = np.array([pred_return_val, last_usd, last_cpi]).reshape(1, 1, 3)
        current_window = np.append(current_window[:, 1:, :], next_step, axis=1)

    return f"✨ Result for {target_date.date()}\n💰 Predicted Gold Price: {predicted_price:.2f} INR"

print(get_momentum_prediction_fixed(target_date))

✨ Result for 2026-02-01
💰 Predicted Gold Price: 136905.51 INR


In [16]:
from sklearn.metrics import mean_absolute_error, r2_score

# Define 'split' for train-test split
split = int(len(X) * 0.8) # Assuming 80% for training and 20% for testing

# 1. Get predictions for the Test Set
y_pred_scaled = model_lstm.predict(X[split:], verbose=0)

# 2. Inverse scale the predicted returns and actual returns
def inverse_only_gold(scaled_val):
    dummy = np.zeros((len(scaled_val), 3))
    dummy[:, 0] = scaled_val.flatten()
    return scaler.inverse_transform(dummy)[:, 0]

actual_returns = inverse_only_gold(y[split:])
pred_returns = inverse_only_gold(y_pred_scaled)

# 3. Convert returns back to Prices to see the real-world error
# We'll use the price from the start of the test period
base_price = df['Price_Gold'].iloc[split + seq_length]
actual_prices = base_price * np.exp(np.cumsum(actual_returns))
pred_prices = base_price * np.exp(np.cumsum(pred_returns))

# 4. Final Metrics
mae = mean_absolute_error(actual_prices, pred_prices)
mape = np.mean(np.abs((actual_prices - pred_prices) / actual_prices)) * 100
accuracy = 100 - mape

print(f"📊 --- LSTM Model Performance ---")
print(f"Mean Absolute Error: {mae:.2f} INR")
print(f"Model Accuracy: {accuracy:.2f}%")

📊 --- LSTM Model Performance ---
Mean Absolute Error: 3434.24 INR
Model Accuracy: 95.56%
